# Coleta Partidas Dota 2

Para conseguirmos coletar sem repetir a coleta, use o arquivo de ids de partida disponível em: [link futuro]()

Para começar, vamos fazer uma conta de padaria:

- podemos fazer 3000 requests por dia por IP
- as requests são contabilizadas por chamadas bem sucedidas da api
- no melhor caso teremos que fazer 1 request por partida
- no pior caso temos que fazer 3 requests: uma para obter a partida, outra para parsear e mais uma para pegar a partida parseada.

vamos coletar com 10 ips (contas diferentesno colab)

altere o valor ID_por_IP na celula "Carrega lista de ids para coletar (COLOQUE SEU 'ID' aqui)"


In [1]:
#@title Dependências
%pip install --upgrade gdown

  Using cached gdown-6.1.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached soupsieve-2.8.4-py3-none-any.whl.metadata (4.6 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-6.1.0-py3-none-any.whl (19 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached filelock-3.29.4-py3-none-any.whl (42 kB)
Using cached tqdm-4.68.3-py3-none-any.whl (78 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
Using cached soupsieve-2.8.4-py3-none-any.whl (37 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
#@title Imports

from IPython.core.display import json
import requests
import pandas as pd
import time
import datetime
import json
import os
import gdown
import shutil


In [2]:
#@title Funções

class NumberAttemptsExceeded(Exception):
  pass

def realiza_request(url, method='get', tentativa=0):
  try:
    if tentativa>=5: raise NumberAttemptsExceeded("Tentou 5 vezes, vamos parar e salvar tudo")
    match method.lower():
      case 'get':
        response = requests.get(url)
      case 'post':
        response = requests.post(url)

    # Verifica se a requisição foi bem sucedida (Status 200)
    response.raise_for_status()
    json_response = response.json()

    if not json_response:
        print("Sem resposta")
        return 404
    return json_response
  except requests.exceptions.RequestException as e:
    print(f"Erro ao acessar a API: {e}\ntentativa: {tentativa}")
    if e.response is not None: # prevê erro de rede (descobri conectado na rede da faculdade)
      if e.response.status_code in [429, 500]:
        time.sleep(65)
        return realiza_request(url, method, tentativa+1)
      else:
        return e.response.status_code
  except NumberAttemptsExceeded as e:
    print(f"Erro ao acessar a API: {e}")
    return 429
  except Exception as e:
    print(f"Erro ao acessar a API: {e}")
    return None

def matches(match_id):
    return realiza_request(f"https://api.opendota.com/api/matches/{match_id}")

def parsear_partida(match_id):
   print(f"parseando id {match_id}")

   resposta = realiza_request(f"https://api.opendota.com/api/request/{match_id}", method='post')

   if resposta in [None, 404, 429]:
      return resposta

   jobId = resposta['job']['jobId']
   time.sleep(1)
   job_em_andamento = True
   print("Esperando job acabar", end="")

   while job_em_andamento:
      resposta = realiza_request(f"https://api.opendota.com/api/request/{jobId}")

      if resposta and 'progress' in str(resposta):
         print(".", end="", flush=True)
         time.sleep(5)
      elif resposta == 429:
        NumberAttemptsExceeded("Tentou 5 vezes, vamos parar e salvar tudo")
      else:
         # Qualquer outra resposta (404, None, ou sucesso) para o job
         job_em_andamento = False
         print()

def get_matches(ids):
  ids_lidos = []
  ids_com_erro = []
  indice=0

  try:
    for id in ids:
      info = matches(id)

      if info == None:
        raise Exception()
      if info == 404:
        print("id com erro 404 ou sem resposta:", id)
        ids_com_erro.append(id)
        continue

      chat = info.get('chat')

      if chat is None:
        print(f"O chat não está disponível para a partida {id} (pode não ter sido parseada).")
        parsear_partida(id)
        info = matches(id)
        chat = info.get('chat')

      with open(f'matches_local/id{id}.json', 'w', encoding='utf-8') as arquivo:
        json.dump(info, arquivo, ensure_ascii=False, indent=2)
      indice+=1
      ids_lidos.append(int(id))

  except Exception as e:
    print(f"erro no id {id} de indice {indice}")
    print(e)
  finally:
    return ids_lidos, ids_com_erro

In [3]:
#@title gambiarra

def idscoletadoscemporcentoatualizado(var="é rum de aturar"):
  file_id = "1BcMLUHoHG8_mbNUEFqhwZV5WWWRmISSF"
  url = f'https://drive.google.com/uc?id={file_id}'
  output = 'partidas_ja_coletadas.json'
  gdown.download(url, output, quiet=False)
  with open('partidas_ja_coletadas.json', 'r') as f:
    partidas = json.load(f)
  return partidas

In [14]:
#@title Carrega lista de ids para coletar (COLOQUE SEU 'ID' aqui)

file_id = "12h2mtHvb5PbaGdtygKyisCN7ZSyCFTsS"
url = f'https://drive.google.com/uc?id={file_id}'
output = 'onlyMatchPlayerID.csv'
gdown.download(url, output, quiet=False)

ID_por_IP = 2 # COLOQUE SEU NÚMERO DE [0..10] AQUI

df_matches = pd.read_csv("onlyMatchPlayerID.csv")
df_matches = df_matches[['match_id']].drop_duplicates()
cem_mil_partidas = df_matches.sample(n=100_000, random_state=42, ignore_index=True) # Não mude o random_state por nada nessa vida

# [ !ATENÇÃO! ] : gambiarra a frente
ids_ja_coletados = idscoletadoscemporcentoatualizado("é ruim de aturar")
para_coletar = list(cem_mil_partidas[cem_mil_partidas["match_id"].index % 10 == ID_por_IP]["match_id"])

para_coletar = list(set(para_coletar) - set(ids_ja_coletados))

os.makedirs("matches_local", exist_ok=True)

with open("matches_local/partidas_coletadas.json", "a", encoding='utf-8') as arquivo:
  arquivo.write("[")



Downloading...
From (original): https://drive.google.com/uc?id=12h2mtHvb5PbaGdtygKyisCN7ZSyCFTsS
From (redirected): https://drive.google.com/uc?id=12h2mtHvb5PbaGdtygKyisCN7ZSyCFTsS&confirm=t&uuid=39765636-d38b-407e-9f3e-f811c8cded77
To: /home/bradachi/Documentos/gitpath/dota2-chat-dataset/onlyMatchPlayerID.csv
100%|██████████| 275M/275M [00:22<00:00, 12.0MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1BcMLUHoHG8_mbNUEFqhwZV5WWWRmISSF
To: /home/bradachi/Documentos/gitpath/dota2-chat-dataset/partidas_ja_coletadas.json
100%|██████████| 139k/139k [00:00<00:00, 882kB/s]


In [11]:
para_coletar = list(cem_mil_partidas[cem_mil_partidas["match_id"].index % 10 == ID_por_IP]["match_id"])

para_coletar = list(set(para_coletar) - set(ids_ja_coletados))

In [15]:
#@title Continuar de onde parou (NÃO EXECUTE ISSO NA PRIMEIRA VEZ)
# atualiza a lista para continuar de onde parou com base no ultimo ID
ultimo_id = 2455795100 #ultimo ID coletado

ultimo_index = para_coletar.index(ultimo_id)
ultimo_valor = para_coletar[ultimo_index]
para_coletar = para_coletar[ultimo_index+1:]
print(ultimo_index, ultimo_valor, "proximo:", para_coletar[0])

9171 2455795100 proximo: 2203350431


In [16]:
#@title Executar a coleta e title Salvar a coleta no pc

try:
  ids_lidos, ids_com_erro = get_matches(para_coletar)
  ids_coletados = {"sucesso": ids_lidos, "erro": ids_com_erro}
except Exception as e:
  print(f"[ ERRO ] {e}")
finally:
  with open('matches_local/partidas_coletadas.json', 'a', encoding='utf-8') as arquivo:
    json.dump(ids_coletados, arquivo, ensure_ascii=False, indent=2)
    arquivo.write(",")

  # shutil.move("/content/partidas_coletadas.json", "/content/matches/partidas_coletadas.json")
  # arquivo_zip = shutil.make_archive('/content/matches', 'zip', '/content/matches')
  # files.download(arquivo_zip)

O chat não está disponível para a partida 2203350431 (pode não ter sido parseada).
parseando id 2203350431
Esperando job acabar
O chat não está disponível para a partida 5139789219 (pode não ter sido parseada).
parseando id 5139789219
Esperando job acabar
O chat não está disponível para a partida 5134153125 (pode não ter sido parseada).
parseando id 5134153125
Esperando job acabar
O chat não está disponível para a partida 7606072745 (pode não ter sido parseada).
parseando id 7606072745
Esperando job acabar
O chat não está disponível para a partida 4950390188 (pode não ter sido parseada).
parseando id 4950390188
Esperando job acabar
O chat não está disponível para a partida 8636757423 (pode não ter sido parseada).
parseando id 8636757423
Esperando job acabarSem resposta

O chat não está disponível para a partida 7028569525 (pode não ter sido parseada).
parseando id 7028569525
Esperando job acabar
O chat não está disponível para a partida 7963768247 (pode não ter sido parseada).
parseand